In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
caminho_base = Path(r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade")

pastas_anos = [pasta for pasta in caminho_base.iterdir()]
d_list = []

for ano in pastas_anos[:-1]:
    arquivos = list(ano.glob("*.csv"))
    for arquivo in arquivos:
        try:
            df = pd.read_csv(arquivo, sep=";", encoding="utf-8", engine="python")
            d_list.append(df)
            # df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
            # df.to_csv(arquivo, index=False, encoding="utf-8", sep=";")
        except Exception as e:
            display(e)


# Tabelas dia de Chuva

In [9]:
import os
import re
from pathlib import Path
import pandas as pd

# Caminho da pasta principal
BASE_DIR = Path(
    r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade"
)
OUTPUT_FILE = BASE_DIR / "DIM_QTDE_CHUVA.csv"


def parse_hora(val):
    """Extrai a hora inteira (0-23) de formatos '00:00', '0000 UTC', '12:00', etc."""
    if pd.isna(val):
        return None
    val_str = str(val).strip().upper()
    digits = re.findall(r"\d+", val_str)
    if not digits:
        return None

    if ":" in val_str:
        return int(digits[0])
    else:
        h_str = digits[0]
        if len(h_str) == 4:
            return int(h_str[:2])
        elif len(h_str) in (1, 2):
            return int(h_str)
        elif len(h_str) == 3:
            return int(h_str[:1])
    return None


def processar_arquivo_inmet(caminho_arquivo, ano_subpasta):
    """Lê e padroniza arquivos do INMET tratando variações de colunas, datas e horas."""
    encoding = "latin-1"

    # Encontra a linha de cabeçalho (caso existam metadados da estação no topo)
    skiprows = 0
    with open(caminho_arquivo, "r", encoding=encoding, errors="ignore") as f:
        for idx, line in enumerate(f):
            if idx > 25:
                break
            if re.search(r"(DATA|Data|DT_MEDICAO)", line, re.IGNORECASE):
                skiprows = idx
                break

    try:
        df = pd.read_csv(
            caminho_arquivo,
            sep=";",
            skiprows=skiprows,
            encoding=encoding,
            decimal=",",
            engine="python",
        )
    except Exception as e:
        print(f"Erro ao ler {caminho_arquivo.name}: {e}")
        return None

    # Limpeza dos nomes das colunas
    df.columns = [str(c).strip() for c in df.columns]

    # Mapeamento dinâmico via Expressões Regulares
    col_data = next((c for c in df.columns if re.search(r"DATA|DT_MEDICAO", c, re.IGNORECASE)), None)
    col_hora = next((c for c in df.columns if re.search(r"HORA|HR_MEDICAO", c, re.IGNORECASE)), None)
    col_chuva = next((c for c in df.columns if re.search(r"PRECIPITA", c, re.IGNORECASE)), None)
    col_umidade = next((c for c in df.columns if re.search(r"UMIDADE RELATIVA DO AR", c, re.IGNORECASE)), None)

    if not all([col_data, col_hora, col_chuva, col_umidade]):
        print(f"Aviso: Colunas essenciais não identificadas em {caminho_arquivo.name}")
        return None

    # Parsing de Data -> Extração de DIA e ANO
    dt_series = pd.to_datetime(
        df[col_data].astype(str).str.replace("/", "-"),
        errors="coerce",
    )

    df_temp = pd.DataFrame()
    df_temp["DIA"] = dt_series.dt.day
    df_temp["ANO"] = dt_series.dt.year.fillna(int(ano_subpasta))
    df_temp["HORA"] = df[col_hora].apply(parse_hora)

    # Tratamento numérico de PRECIPITAÇÃO_TOTAL_HORA e UMIDADE_RELATIVA_AR
    mapeamento_colunas = {
        "PRECIPITAÇÃO_TOTAL_HORA": col_chuva,
        "UMIDADE_RELATIVA_AR": col_umidade,
    }

    for target_col, orig_col in mapeamento_colunas.items():
        series = df[orig_col]
        if series.dtype == object:
            series = series.astype(str).str.replace(",", ".")
        num_series = pd.to_numeric(series, errors="coerce")

        # Invalidação de códigos de erro INMET (-9999)
        df_temp[target_col] = num_series.apply(
            lambda x: None if (pd.notna(x) and x < -100) else x
        )

    # Filtragem de linhas sem chaves válidas
    df_temp = df_temp.dropna(subset=["DIA", "ANO", "HORA"])

    # Conversão explícita de tipos inteiros
    df_temp["DIA"] = df_temp["DIA"].astype(int)
    df_temp["ANO"] = df_temp["ANO"].astype(int)
    df_temp["HORA"] = df_temp["HORA"].astype(int)

    return df_temp


def gerar_dim_qtde_chuva():
    lista_dfs = []

    # Iteração sobre subpastas (ex: 2014, 2018, 2022, 2026)
    for subpasta in BASE_DIR.iterdir():
        if subpasta.is_dir() and subpasta.name.isdigit():
            ano = subpasta.name
            print(f"Processando arquivos da pasta do ano: {ano}...")

            for arquivo in subpasta.glob("*.[cC][sS][vV]"):
                df_proc = processar_arquivo_inmet(arquivo, ano)
                if df_proc is not None and not df_proc.empty:
                    lista_dfs.append(df_proc)

    if not lista_dfs:
        print("Nenhum arquivo válido foi processado.")
        return

    # União de todos os anos e estações
    df_consolidado = pd.concat(lista_dfs, ignore_index=True)

    # Agrupamento para garantia da Chave Primária (HORA, DIA, ANO) do DBML
    # Média entre estações caso existam múltiplas medições na mesma hora/dia/ano
    df_final = (
        df_consolidado.groupby(["HORA", "DIA", "ANO"], as_index=False)
        .agg({
            "PRECIPITAÇÃO_TOTAL_HORA": "mean",
            "UMIDADE_RELATIVA_AR": "mean",
        })
        .round(2)
    )

    # Ordenação das colunas de acordo com a especificação da tabela DIM_QTDE_CHUVA
    colunas_dbml = [
        "DIA",
        "ANO",
        "HORA",
        "PRECIPITAÇÃO_TOTAL_HORA",
        "UMIDADE_RELATIVA_AR",
    ]
    df_final = df_final[colunas_dbml]

    # Exportação final para CSV
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\n[SUCESSO] Tabela gerada: {OUTPUT_FILE}")
    print(f"Total de registros na dimensão: {len(df_final)}")


if __name__ == "__main__":
    gerar_dim_qtde_chuva()

Processando arquivos da pasta do ano: 2014...
Processando arquivos da pasta do ano: 2018...
Processando arquivos da pasta do ano: 2022...

[SUCESSO] Tabela gerada: C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\DIM_QTDE_CHUVA.csv
Total de registros na dimensão: 288


In [33]:
ARQUIVO_CSV = Path(
    r'C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\Quanto Choveu\DIM_QTDE_CHUVA.csv'
)


df = pd.read_csv(ARQUIVO_CSV, sep=',', engine='python')

df['CHOVEU_FORTE'] = (df['PRECIPITAÇÃO_TOTAL_HORA'] > 1)

display(df['CHOVEU_FORTE'].value_counts())

df

# df_ordenado = df.sort_values(["ANO", "DIA", "HORA"], ascending=[True, True, True])

# df_ordenado.to_csv(ARQUIVO_CSV, index=False, encoding='utf-8-sig')

CHOVEU_FORTE
False    270
True      18
Name: count, dtype: int64

,DIA,ANO,HORA,PRECIPITAÇÃO_TOTAL_HORA,UMIDADE_RELATIVA_AR,CHOVEU_FORTE
0,4,2014,0,0.000000,83.176471,False
1,4,2014,1,0.294118,85.705882,False
2,4,2014,2,1.929412,87.352941,True
3,4,2014,3,0.364706,87.941176,False
4,4,2014,4,0.305882,88.058824,False
...,...,...,...,...,...,...
283,30,2022,19,0.000000,58.600000,False
284,30,2022,20,0.000000,59.466667,False
285,30,2022,21,0.000000,63.866667,False
286,30,2022,22,0.000000,72.266667,False


In [24]:
df.columns


Index(['DIA', 'ANO', 'HORA', 'PRECIPITAÇÃO_TOTAL_HORA', 'UMIDADE_RELATIVA_AR'], dtype='str')